# LeetCode 1361 — Validate Binary Tree Nodes bằng Python

Link đề: https://leetcode.com/problems/validate-binary-tree-nodes/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Union-Find, Graph Theory, Binary Tree

Skeleton LeetCode:

```python
class Solution:
    def validateBinaryTreeNodes(self, n: int, leftChild: list[int], rightChild: list[int]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa **cây nhị phân hợp lệ** dưới góc nhìn **đồ thị có hướng**: đúng một gốc (bậc vào 0), mọi nút khác có **đúng một cha**, và **mọi nút đến được từ gốc** (liên thông, không chu trình).
2. Biết vì sao chỉ kiểm tra "bậc vào ≤ 1 và có đúng một gốc" là **chưa đủ** (còn chu trình tách rời).
3. Cài đặt: brute force thử mọi gốc `O(n²)` → **bậc vào + tìm gốc + BFS/DFS** `O(n)` → **Union-Find**.
4. Thành thạo BFS bằng `collections.deque`, đếm số nút đã thăm, phát hiện thăm lại.
5. Tự dựng `TreeNode` từ hai mảng con trái/phải để **hiển thị** cây hợp lệ (level-order).

## 2. Đề bài & ví dụ

Có `n` nút cây nhị phân đánh số `0..n−1`. Nút `i` có con trái `leftChild[i]` và con phải `rightChild[i]` (`−1` nếu không có). Trả về `True` khi và chỉ khi **tất cả** các nút tạo thành **đúng một** cây nhị phân hợp lệ. Các nút không có giá trị, chỉ dùng số hiệu.

**Ví dụ 1:** `n = 4, leftChild = [1,-1,3,-1], rightChild = [2,-1,-1,-1]` → `True` (gốc 0; 0 có con 1, 2; 2 có con trái 3).

**Ví dụ 2:** `n = 4, leftChild = [1,-1,3,-1], rightChild = [2,3,-1,-1]` → `False` (nút 3 có hai cha: 1 và 2).

**Ví dụ 3:** `n = 2, leftChild = [1,0], rightChild = [-1,-1]` → `False` (0 → 1 → 0: chu trình, không có gốc).

**Ràng buộc:** `n == leftChild.length == rightChild.length`; `1 <= n <= 10^4`; `-1 <= leftChild[i], rightChild[i] <= n − 1`.

In [1]:
import random
import time
from collections import deque

examples = [
    (4, [1, -1, 3, -1], [2, -1, -1, -1], True),
    (4, [1, -1, 3, -1], [2, 3, -1, -1], False),
    (2, [1, 0], [-1, -1], False),
]
for n, L, R, want in examples:
    edges = [(i, c) for i in range(n) for c in (L[i], R[i]) if c != -1]
    print(f"n={n}, cạnh cha->con = {edges} -> mong đợi {want}")

n=4, cạnh cha->con = [(0, 1), (0, 2), (2, 3)] -> mong đợi True
n=4, cạnh cha->con = [(0, 1), (0, 2), (1, 3), (2, 3)] -> mong đợi False
n=2, cạnh cha->con = [(0, 1), (1, 0)] -> mong đợi False


## 3. Kiến thức nền

### 3.1 Cây nhị phân như một đồ thị có hướng

Mỗi cặp "`i` có con `c`" là một **cạnh có hướng** `i → c`. Một tập nút là **một cây** (có gốc) khi:
1. Có **đúng một** nút không có cha (gốc).
2. Mọi nút khác có **đúng một** cha (bậc vào = 1).
3. Từ gốc đi được tới **mọi** nút (khi đó tự động không có chu trình, và số cạnh = `n − 1`).

In [2]:
def edges_of(n, L, R):
    return [(i, c) for i in range(n) for c in (L[i], R[i]) if c != -1]

n, L, R, _ = examples[0]
E = edges_of(n, L, R)
print("cạnh:", E, "| số cạnh:", len(E), "| n - 1 =", n - 1)

cạnh: [(0, 1), (0, 2), (2, 3)] | số cạnh: 3 | n - 1 = 3


### 3.2 Bậc vào (in-degree)

`indeg[c]` = số cạnh đi vào `c` = số cha của `c`. Tính bằng một lần duyệt hai mảng.

In [3]:
def indegrees(n, L, R):
    indeg = [0] * n
    for c in L + R:
        if c != -1:
            indeg[c] += 1
    return indeg

for n, L, R, want in examples:
    print(f"n={n}: indeg = {indegrees(n, L, R)}")

n=4: indeg = [0, 1, 1, 1]
n=4: indeg = [0, 1, 1, 2]
n=2: indeg = [1, 1]


### 3.3 BFS với `deque` và đếm số nút thăm

`deque.popleft()` lấy phần tử đầu `O(1)`. BFS từ gốc: lấy nút, đẩy các con vào hàng đợi. Dùng mảng `seen` để phát hiện **thăm lại** (dấu hiệu nhiều cha hoặc chu trình).

In [4]:
def bfs_order(root, L, R):
    order, q = [], deque([root])
    while q:
        u = q.popleft()
        order.append(u)
        for c in (L[u], R[u]):
            if c != -1:
                q.append(c)
    return order

n, L, R, _ = examples[0]
print("BFS từ 0:", bfs_order(0, L, R))

BFS từ 0: [0, 1, 2, 3]


### 3.4 Union-Find (Disjoint Set Union)

`find(x)` trả về đại diện của nhóm chứa `x`; `union(a, b)` gộp hai nhóm. Nếu nối một cạnh mà hai đầu **đã cùng nhóm** ⇒ tạo chu trình (theo nghĩa vô hướng). Đếm số nhóm còn lại để kiểm tra liên thông.

In [5]:
parent = list(range(5))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return False
    parent[rb] = ra
    return True

print(union(0, 1), union(1, 2), union(0, 2), "<- cạnh thứ ba tạo chu trình")
print("số nhóm:", len({find(i) for i in range(5)}))

True True False <- cạnh thứ ba tạo chu trình
số nhóm: 3


### 3.5 `TreeNode` và dựng cây để hiển thị

Bài cho cây dưới dạng mảng, nhưng để "nhìn" cây hợp lệ ta dựng `TreeNode` (giá trị = số hiệu nút) rồi in **level-order** với `None` cho chỗ trống — giống định dạng LeetCode.

In [6]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree(root, L, R):
    """Dựng TreeNode từ (L, R) — chỉ gọi khi đã biết là cây hợp lệ."""
    nodes = {}
    def make(i):
        if i == -1:
            return None
        node = TreeNode(i)
        node.left, node.right = make(L[i]), make(R[i])
        return node
    return make(root)

def to_level_order(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val)
        q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def build_from_level_order(vals):
    """Hàm ngược: từ mảng level-order (có None) -> (n, leftChild, rightChild), nút đánh số theo giá trị."""
    n = sum(v is not None for v in vals)
    L, R = [-1] * n, [-1] * n
    idx = 1
    for v in vals:
        if v is None:
            continue
        if idx < len(vals):
            if vals[idx] is not None: L[v] = vals[idx]
            idx += 1
        if idx < len(vals):
            if vals[idx] is not None: R[v] = vals[idx]
            idx += 1
    return n, L, R

n, L, R, _ = examples[0]
print("level-order:", to_level_order(build_tree(0, L, R)))
print("dựng ngược :", build_from_level_order([0, 1, 2, None, None, 3]))

level-order: [0, 1, 2, None, None, 3]
dựng ngược : (4, [1, -1, 3, -1], [2, -1, -1, -1])


## 4. Ý tưởng nền tảng / trực giác

Phân tích tay ba ví dụ:

| Ví dụ | Cạnh | indeg | Nút bậc vào 0 | Vấn đề |
|---|---|---|---|---|
| 1 | 0→1, 0→2, 2→3 | [0,1,1,1] | {0} | không — BFS từ 0 thăm 4 nút ⇒ True |
| 2 | 0→1, 0→2, 1→3, 2→3 | [0,1,1,**2**] | {0} | nút 3 có 2 cha ⇒ False |
| 3 | 0→1, 1→0 | [1,1] | **∅** | không có gốc (chu trình) ⇒ False |

Một trường hợp "khó" tự dựng: `n = 4, leftChild = [1,0,3,-1], rightChild = [-1,-1,-1,-1]` — cạnh `0→1, 1→0, 2→3`. Bậc vào `[1,1,0,1]`: mọi nút ≤ 1 cha, có **đúng một** gốc (nút 2)… nhưng `0 ↔ 1` là một chu trình **tách rời**. BFS từ 2 chỉ thăm `{2, 3}` ⇒ False. Bước liên thông là **bắt buộc**.

In [7]:
tricky = (4, [1, 0, 3, -1], [-1, -1, -1, -1], False)
n, L, R, _ = tricky
indeg = indegrees(n, L, R)
roots = [i for i in range(n) if indeg[i] == 0]
print("indeg:", indeg, "| gốc:", roots)
print("BFS từ gốc thăm:", bfs_order(roots[0], L, R), "-> thiếu nút => False")

indeg: [1, 1, 0, 1] | gốc: [2]
BFS từ gốc thăm: [2, 3] -> thiếu nút => False


## 5. Cách 1 — Brute Force / baseline: thử từng nút làm gốc

**Ý tưởng:** với mỗi `r` từ `0` đến `n−1`, chạy DFS/BFS từ `r` với tập `seen`. Nếu gặp lại một nút đã thăm ⇒ không phải cây từ gốc `r` (nút có 2 cha hoặc chu trình). Nếu không gặp lại và thăm đủ `n` nút ⇒ True. Không gốc nào thành công ⇒ False.

In [8]:
def validate_brute(n, L, R):
    for r in range(n):
        seen = [False] * n
        seen[r] = True
        stack, count, ok = [r], 1, True
        while stack and ok:
            u = stack.pop()
            for c in (L[u], R[u]):
                if c == -1:
                    continue
                if seen[c]:          # thăm lại -> không phải cây
                    ok = False
                    break
                seen[c] = True
                count += 1
                stack.append(c)
        if ok and count == n:
            return True
    return False

for n, L, R, want in examples + [tricky]:
    print(f"n={n} L={L} R={R} -> {validate_brute(n, L, R)} (mong đợi {want})")

n=4 L=[1, -1, 3, -1] R=[2, -1, -1, -1] -> True (mong đợi True)
n=4 L=[1, -1, 3, -1] R=[2, 3, -1, -1] -> False (mong đợi False)
n=2 L=[1, 0] R=[-1, -1] -> False (mong đợi False)
n=4 L=[1, 0, 3, -1] R=[-1, -1, -1, -1] -> False (mong đợi False)


**Độ phức tạp:** `n` lần duyệt, mỗi lần `O(n)` ⇒ `O(n²)`; với `n = 10^4` là `10^8` bước — quá chậm trong Python. Bộ nhớ `O(n)`.

**Nút thắt:** ta thử cả những nút **chắc chắn không thể là gốc** (nút có cha). Gốc của cây là nút **duy nhất** có bậc vào 0 ⇒ tìm được ngay bằng một lần đếm.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** điều kiện "là một cây" ⇔ ba kiểm tra rẻ:

1. **Bậc vào:** mọi nút có `indeg ≤ 1`. (Nếu nút có 2 cha ⇒ False ngay.)
2. **Gốc duy nhất:** đúng **một** nút có `indeg == 0`. (0 gốc ⇒ có chu trình chứa mọi nút; ≥ 2 gốc ⇒ rừng.)
3. **Liên thông:** BFS/DFS từ gốc thăm được **đủ `n` nút**.

**Quy tắc nếu… thì…:**
- Nếu `indeg[c]` vượt 1 thì trả False.
- Nếu số nút bậc vào 0 khác 1 thì trả False.
- Nếu BFS từ gốc thăm `< n` nút thì trả False (có chu trình tách rời).

**Quan sát thêm:** khi (1) và (2) đúng, BFS từ gốc **không thể** thăm lại nút nào (mỗi nút có ≤ 1 cha, gốc không có cha) ⇒ không cần mảng `seen`, BFS tự dừng. Ngoài ra (1) + (2) ⇒ số cạnh = `n − 1`.

**Góc nhìn Union-Find:** duyệt các cạnh `p → c`; nếu `c` đã có cha ⇒ False; nếu `p`, `c` đã cùng nhóm ⇒ chu trình ⇒ False; cuối cùng phải còn **đúng 1 nhóm**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Bậc vào + gốc duy nhất + BFS

**Giải thích từng đoạn:**
1. Đếm `indeg`; gặp `indeg > 1` ⇒ False.
2. Lấy danh sách gốc; nếu không đúng 1 ⇒ False.
3. BFS từ gốc, đếm số nút; so với `n`.

In [9]:
def validate_bfs(n, L, R):
    indeg = [0] * n
    for c in L + R:
        if c != -1:
            indeg[c] += 1
            if indeg[c] > 1:
                return False
    roots = [i for i in range(n) if indeg[i] == 0]
    if len(roots) != 1:
        return False
    q, count = deque([roots[0]]), 0
    while q:
        u = q.popleft()
        count += 1
        for c in (L[u], R[u]):
            if c != -1:
                q.append(c)
    return count == n

for n, L, R, want in examples + [tricky]:
    print(f"n={n} -> {validate_bfs(n, L, R)} (mong đợi {want})")

n=4 -> True (mong đợi True)
n=4 -> False (mong đợi False)
n=2 -> False (mong đợi False)
n=4 -> False (mong đợi False)


Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def validateBinaryTreeNodes(self, n: int, leftChild: list[int], rightChild: list[int]) -> bool:
        indeg = [0] * n
        for c in leftChild + rightChild:
            if c != -1:
                indeg[c] += 1
                if indeg[c] > 1:
                    return False
        roots = [i for i in range(n) if indeg[i] == 0]
        if len(roots) != 1:
            return False
        q, count = deque([roots[0]]), 0
        while q:
            u = q.popleft()
            count += 1
            for c in (leftChild[u], rightChild[u]):
                if c != -1:
                    q.append(c)
        return count == n

print(Solution().validateBinaryTreeNodes(4, [1, -1, 3, -1], [2, -1, -1, -1]))

True


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Union-Find

**Giải thích từng đoạn:**
1. `has_parent[c]` đánh dấu nút đã có cha; gặp lần hai ⇒ False.
2. `find(p) == find(c)` ⇒ cạnh này khép một chu trình ⇒ False. Ngược lại gộp, `components −= 1`.
3. Cuối cùng `components == 1` ⇔ liên thông (và vì không chu trình, đó là một cây có `n − 1` cạnh).

Vì sao một cây vô hướng + mỗi nút ≤ 1 cha là cây có gốc? `n` nút, `n − 1` cạnh, mỗi cạnh cho một nút đúng một cha ⇒ đúng `n − 1` nút có cha ⇒ đúng một gốc; liên thông + không chu trình ⇒ mọi nút đến được từ gốc theo đường cha → con duy nhất.

In [11]:
def validate_dsu(n, L, R):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    has_parent = [False] * n
    components = n
    for p in range(n):
        for c in (L[p], R[p]):
            if c == -1:
                continue
            if has_parent[c]:
                return False                  # hai cha
            has_parent[c] = True
            rp, rc = find(p), find(c)
            if rp == rc:
                return False                  # chu trình
            parent[rc] = rp
            components -= 1
    return components == 1

for n, L, R, want in examples + [tricky]:
    print(f"n={n} -> {validate_dsu(n, L, R)} (mong đợi {want})")

n=4 -> True (mong đợi True)
n=4 -> False (mong đợi False)
n=2 -> False (mong đợi False)
n=4 -> False (mong đợi False)


Code nộp LeetCode (Cách 3):

In [12]:
class Solution:
    def validateBinaryTreeNodes(self, n: int, leftChild: list[int], rightChild: list[int]) -> bool:
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        has_parent = [False] * n
        components = n
        for p in range(n):
            for c in (leftChild[p], rightChild[p]):
                if c == -1:
                    continue
                if has_parent[c]:
                    return False
                has_parent[c] = True
                rp, rc = find(p), find(c)
                if rp == rc:
                    return False
                parent[rc] = rp
                components -= 1
        return components == 1

print(Solution().validateBinaryTreeNodes(2, [1, 0], [-1, -1]))

False


**Độ phức tạp:** `O(n · α(n))` ≈ `O(n)`; bộ nhớ `O(n)`.

### 7.3 Cách 4 — DFS đệ quy đếm kích thước cây con (sau khi kiểm tra bậc vào)

**Giải thích:** sau bước (1) + (2), dùng DFS trả về **số nút trong cây con**: `size(u) = 1 + size(L[u]) + size(R[u])`, `size(−1) = 0`. Hợp lệ ⇔ `size(root) == n`. Để tránh giới hạn đệ quy của Python với cây "dây xích" sâu `10^4`, ta dùng **stack tường minh** mô phỏng đệ quy.

In [13]:
def validate_dfs_size(n, L, R):
    indeg = [0] * n
    for c in L + R:
        if c != -1:
            indeg[c] += 1
    if any(d > 1 for d in indeg) or indeg.count(0) != 1:
        return False
    root = indeg.index(0)
    size = 0
    stack = [root]
    while stack:                       # DFS lặp, an toàn với cây sâu
        u = stack.pop()
        size += 1
        if L[u] != -1: stack.append(L[u])
        if R[u] != -1: stack.append(R[u])
    return size == n

for n, L, R, want in examples + [tricky]:
    print(f"n={n} -> {validate_dfs_size(n, L, R)} (mong đợi {want})")

n=4 -> True (mong đợi True)
n=4 -> False (mong đợi False)
n=2 -> False (mong đợi False)
n=4 -> False (mong đợi False)


## 8. TRACE từng bước

### 8.1 Trace Cách 2 trên ví dụ chu trình tách rời (`tricky`)

| Bước | Trạng thái |
|---|---|
| indeg | `[1, 1, 0, 1]` — không nút nào > 1 |
| gốc | `[2]` — đúng một |
| BFS | thăm 2 → đẩy 3; thăm 3 |
| count | 2 ≠ 4 ⇒ **False** |

In [14]:
def trace_bfs(n, L, R):
    indeg = [0] * n
    for c in L + R:
        if c != -1:
            indeg[c] += 1
    print("indeg =", indeg)
    if max(indeg) > 1:
        print("có nút nhiều cha:", [i for i in range(n) if indeg[i] > 1], "-> False")
        return False
    roots = [i for i in range(n) if indeg[i] == 0]
    print("gốc =", roots)
    if len(roots) != 1:
        print("số gốc khác 1 -> False")
        return False
    q, count = deque([roots[0]]), 0
    while q:
        u = q.popleft()
        count += 1
        kids = [c for c in (L[u], R[u]) if c != -1]
        q.extend(kids)
        print(f"  thăm {u}, con {kids}, hàng đợi {list(q)}, đã thăm {count}")
    print(f"count={count}, n={n} ->", count == n)
    return count == n

for case in examples + [tricky]:
    print("=== n =", case[0], "L =", case[1], "R =", case[2])
    trace_bfs(*case[:3])

=== n = 4 L = [1, -1, 3, -1] R = [2, -1, -1, -1]
indeg = [0, 1, 1, 1]
gốc = [0]
  thăm 0, con [1, 2], hàng đợi [1, 2], đã thăm 1
  thăm 1, con [], hàng đợi [2], đã thăm 2
  thăm 2, con [3], hàng đợi [3], đã thăm 3
  thăm 3, con [], hàng đợi [], đã thăm 4
count=4, n=4 -> True
=== n = 4 L = [1, -1, 3, -1] R = [2, 3, -1, -1]
indeg = [0, 1, 1, 2]
có nút nhiều cha: [3] -> False
=== n = 2 L = [1, 0] R = [-1, -1]
indeg = [1, 1]
gốc = []
số gốc khác 1 -> False
=== n = 4 L = [1, 0, 3, -1] R = [-1, -1, -1, -1]
indeg = [1, 1, 0, 1]
gốc = [2]
  thăm 2, con [3], hàng đợi [3], đã thăm 1
  thăm 3, con [], hàng đợi [], đã thăm 2
count=2, n=4 -> False


### 8.2 Trace Cách 3 (Union-Find) trên ví dụ 2

In [15]:
def trace_dsu(n, L, R):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    has_parent = [False] * n
    comp = n
    for p in range(n):
        for c in (L[p], R[p]):
            if c == -1:
                continue
            if has_parent[c]:
                print(f"cạnh {p}->{c}: {c} đã có cha -> False")
                return False
            has_parent[c] = True
            rp, rc = find(p), find(c)
            if rp == rc:
                print(f"cạnh {p}->{c}: cùng nhóm {rp} -> chu trình -> False")
                return False
            parent[rc] = rp
            comp -= 1
            print(f"cạnh {p}->{c}: gộp nhóm {rc} vào {rp}, parent={parent}, số nhóm={comp}")
    print("số nhóm cuối:", comp, "->", comp == 1)
    return comp == 1

trace_dsu(*examples[1][:3])
print()
trace_dsu(*examples[0][:3])

cạnh 0->1: gộp nhóm 1 vào 0, parent=[0, 0, 2, 3], số nhóm=3
cạnh 0->2: gộp nhóm 2 vào 0, parent=[0, 0, 0, 3], số nhóm=2
cạnh 1->3: gộp nhóm 3 vào 0, parent=[0, 0, 0, 0], số nhóm=1
cạnh 2->3: 3 đã có cha -> False

cạnh 0->1: gộp nhóm 1 vào 0, parent=[0, 0, 2, 3], số nhóm=3
cạnh 0->2: gộp nhóm 2 vào 0, parent=[0, 0, 0, 3], số nhóm=2
cạnh 2->3: gộp nhóm 3 vào 0, parent=[0, 0, 0, 0], số nhóm=1
số nhóm cuối: 1 -> True


True

## 9. Code nộp LeetCode

In [16]:
from collections import deque

class Solution:
    def validateBinaryTreeNodes(self, n: int, leftChild: list[int], rightChild: list[int]) -> bool:
        indeg = [0] * n
        for c in leftChild + rightChild:
            if c != -1:
                indeg[c] += 1
                if indeg[c] > 1:              # một nút có hai cha
                    return False
        roots = [i for i in range(n) if indeg[i] == 0]
        if len(roots) != 1:                   # không có gốc hoặc nhiều gốc
            return False
        q, count = deque([roots[0]]), 0
        while q:                              # BFS kiểm tra liên thông
            u = q.popleft()
            count += 1
            for c in (leftChild[u], rightChild[u]):
                if c != -1:
                    q.append(c)
        return count == n

sol = Solution()
for n, L, R, want in examples + [tricky]:
    assert sol.validateBinaryTreeNodes(n, L, R) == want
    print(f"validateBinaryTreeNodes({n}, {L}, {R}) = {sol.validateBinaryTreeNodes(n, L, R)}")
print("Cây ví dụ 1 (level-order):", to_level_order(build_tree(0, examples[0][1], examples[0][2])))

validateBinaryTreeNodes(4, [1, -1, 3, -1], [2, -1, -1, -1]) = True
validateBinaryTreeNodes(4, [1, -1, 3, -1], [2, 3, -1, -1]) = False
validateBinaryTreeNodes(2, [1, 0], [-1, -1]) = False
validateBinaryTreeNodes(4, [1, 0, 3, -1], [-1, -1, -1, -1]) = False
Cây ví dụ 1 (level-order): [0, 1, 2, None, None, 3]


## 10. Vì sao thuật toán đúng

**(⇒) Nếu là cây hợp lệ** thì: gốc có bậc vào 0, mọi nút khác có đúng một cha ⇒ kiểm tra (1), (2) qua; mọi nút đến được từ gốc ⇒ BFS đếm đủ `n` ⇒ True.

**(⇐) Nếu thuật toán trả True:** mọi nút có ≤ 1 cha, đúng một nút `r` không có cha, và BFS từ `r` thăm `n` lần.
- BFS không thăm lại nút nào: một nút được đẩy vào hàng đợi chỉ khi cha của nó được thăm; mỗi nút có ≤ 1 cha, và `r` không có cha ⇒ mỗi nút được đẩy **tối đa một lần** (quy nạp theo thứ tự thăm). Vậy `count == n` ⇒ **mọi** nút được thăm đúng một lần.
- Mỗi nút `≠ r` được thăm qua đúng một cha, đường từ `r` tới nó là duy nhất ⇒ không chu trình, đúng cấu trúc cây; mỗi nút có ≤ 2 con theo định nghĩa mảng ⇒ cây **nhị phân**.

**Union-Find:** nếu không có cạnh nào khép chu trình (vô hướng) và còn 1 nhóm thì đồ thị vô hướng là cây `n − 1` cạnh; kết hợp "mỗi nút ≤ 1 cha" ⇒ đúng `n − 1` nút có cha ⇒ một gốc, và hướng cạnh đều từ cha xuống con ⇒ cây có gốc hợp lệ. Ngược lại, cây hợp lệ không bao giờ kích hoạt các điều kiện trả False.

## 11. Phân tích Time / Space

Gọi `n` = số nút (số cạnh ≤ `2n`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi gốc | `O(n²)` | `O(n)` | baseline |
| 2. Bậc vào + gốc + BFS | `O(n)` | `O(n)` | lời giải chuẩn |
| 3. Union-Find | `O(n α(n))` | `O(n)` | xử lý từng cạnh, dừng sớm |
| 4. Bậc vào + DFS đếm kích thước | `O(n)` | `O(n)` | stack tường minh, an toàn cây sâu |

Trường hợp "dừng sớm": Cách 2/3 trả False ngay khi phát hiện nút hai cha.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
METHODS = {
    "brute": validate_brute,
    "bfs": validate_bfs,
    "dsu": validate_dsu,
    "dfs_size": validate_dfs_size,
    "submit": Solution().validateBinaryTreeNodes,
}
cases = examples + [tricky] + [
    (1, [-1], [-1], True),                    # một nút
    (1, [0], [-1], False),                    # tự trỏ vào chính nó
    (2, [-1, -1], [-1, -1], False),           # hai nút rời (rừng)
    (2, [-1, 0], [-1, -1], True),             # gốc là nút 1, không phải 0
    (3, [1, -1, -1], [1, -1, -1], False),     # con trái = con phải = 1 (bậc vào 2)
    (3, [1, 2, 0], [-1, -1, -1], False),      # chu trình 0->1->2->0
    (4, [3, -1, 1, -1], [-1, -1, 0, -1], True),   # gốc 2
    (6, [1, -1, -1, 4, -1, -1], [2, -1, -1, 5, -1, -1], False),  # hai cây rời
]
for n, L, R, want in cases:
    for name, f in METHODS.items():
        got = f(n, L, R)
        assert got == want, (name, n, L, R, got, want)
print(f"{len(cases)} ca × {len(METHODS)} cách: đúng hết")

12 ca × 5 cách: đúng hết


### 12.2 Sinh cây hợp lệ ngẫu nhiên + đột biến (seed cố định)

Sinh cây nhị phân ngẫu nhiên (dựng bằng `TreeNode` từ mảng level-order, rồi **đổi nhãn** ngẫu nhiên) ⇒ đáp án chắc chắn True. Sau đó **đột biến** một con trỏ ngẫu nhiên ⇒ đáp án có thể True/False, đối chiếu với brute force.

In [18]:
def random_valid_tree(rng, n):
    """Trả về (n, L, R) là một cây nhị phân hợp lệ với nhãn ngẫu nhiên."""
    perm = list(range(n)); rng.shuffle(perm)
    L, R = [-1] * n, [-1] * n
    free = [(perm[0], 0), (perm[0], 1)]          # các "khe" con còn trống
    for k in range(1, n):
        i = rng.randrange(len(free))
        p, side = free.pop(i)
        (L if side == 0 else R)[p] = perm[k]
        free += [(perm[k], 0), (perm[k], 1)]
    return n, L, R

rng = random.Random(1361)
for _ in range(300):
    n, L, R = random_valid_tree(rng, rng.randint(1, 12))
    for name, f in METHODS.items():
        assert f(n, L, R) is True, (name, n, L, R)
    # kiểm tra vòng: dựng TreeNode -> level-order -> dựng lại -> vẫn hợp lệ
    root = [i for i in range(n) if i not in L + R][0]
    n2, L2, R2 = build_from_level_order(to_level_order(build_tree(root, L, R)))
    assert n2 == n and validate_bfs(n2, L2, R2)
print("300 cây hợp lệ ngẫu nhiên: mọi cách trả True")

stat = {True: 0, False: 0}
for _ in range(1500):
    n, L, R = random_valid_tree(rng, rng.randint(1, 9))
    for _ in range(rng.randint(1, 2)):           # đột biến 1-2 con trỏ
        arr = rng.choice([L, R])
        arr[rng.randrange(n)] = rng.randint(-1, n - 1)
    ref = validate_brute(n, L, R)
    stat[ref] += 1
    for name, f in METHODS.items():
        assert f(n, L, R) == ref, (name, n, L, R)
print("1500 đầu vào đột biến: mọi cách khớp brute force | phân bố đáp án:", stat)

300 cây hợp lệ ngẫu nhiên: mọi cách trả True
1500 đầu vào đột biến: mọi cách khớp brute force | phân bố đáp án: {True: 232, False: 1268}


### 12.3 Kích thước lớn nhất `n = 10^4` (cây dây xích sâu và cây ngẫu nhiên)

In [19]:
N = 10**4
chain = (N, list(range(1, N)) + [-1], [-1] * N)             # 0->1->2->...: sâu 10^4
chain_cycle = (N, list(range(1, N)) + [0], [-1] * N)       # thêm cạnh quay về 0
big_tree = random_valid_tree(random.Random(9), N)
for label, case, want in [("dây xích", chain, True), ("dây xích + chu trình", chain_cycle, False), ("cây ngẫu nhiên", big_tree, True)]:
    for name in ["bfs", "dsu", "dfs_size"]:
        st = time.perf_counter()
        got = METHODS[name](*case)
        assert got == want
        print(f"{label:22} {name:8} -> {got} ({time.perf_counter() - st:.3f}s)")

dây xích               bfs      -> True (0.002s)
dây xích               dsu      -> True (0.003s)
dây xích               dfs_size -> True (0.002s)
dây xích + chu trình   bfs      -> False (0.001s)
dây xích + chu trình   dsu      -> False (0.003s)
dây xích + chu trình   dfs_size -> False (0.001s)
cây ngẫu nhiên         bfs      -> True (0.003s)
cây ngẫu nhiên         dsu      -> True (0.004s)
cây ngẫu nhiên         dfs_size -> True (0.002s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra bậc vào và số gốc (quên liên thông)

In [20]:
def wrong_no_connectivity(n, L, R):
    indeg = indegrees(n, L, R)
    return max(indeg) <= 1 and indeg.count(0) == 1

n, L, R, want = tricky
print("sai:", wrong_no_connectivity(n, L, R), "| đúng:", want, "(chu trình 0<->1 tách rời)")

sai: True | đúng: False (chu trình 0<->1 tách rời)


**Vì sao sai:** một chu trình tách rời (`0 ↔ 1`) có mọi nút bậc vào 1, không tạo thêm gốc. **Sửa:** BFS/DFS từ gốc phải thăm đủ `n` nút.

### Lỗi 2 — Giả định gốc luôn là nút 0

In [21]:
def wrong_root_zero(n, L, R):
    seen, q = {0}, deque([0])
    while q:
        u = q.popleft()
        for c in (L[u], R[u]):
            if c != -1:
                if c in seen:
                    return False
                seen.add(c); q.append(c)
    return len(seen) == n

for n, L, R in [(2, [-1, 0], [-1, -1]), (4, [3, -1, 1, -1], [-1, -1, 0, -1])]:
    print(f"n={n} L={L} R={R}: sai cho {wrong_root_zero(n, L, R)}, đúng là {validate_bfs(n, L, R)}")

n=2 L=[-1, 0] R=[-1, -1]: sai cho False, đúng là True
n=4 L=[3, -1, 1, -1] R=[-1, -1, 0, -1]: sai cho False, đúng là True


**Vì sao sai:** đề không nói gốc là 0. **Sửa:** tìm gốc = nút duy nhất có bậc vào 0.

### Lỗi 3 — BFS không kiểm tra bậc vào / thăm lại (lặp vô hạn)

In [22]:
def wrong_infinite(n, L, R, limit=50):
    """Không kiểm tra gì, BFS từ 0; giới hạn số bước để không treo."""
    q, steps = deque([0]), 0
    while q:
        u = q.popleft()
        steps += 1
        if steps > limit:
            return f"> {limit} bước (sẽ chạy mãi)"
        for c in (L[u], R[u]):
            if c != -1:
                q.append(c)
    return steps == n

print("ví dụ 3 (chu trình):", wrong_infinite(2, [1, 0], [-1, -1]))
print("ví dụ 2 (nút 2 cha):", wrong_infinite(4, [1, -1, 3, -1], [2, 3, -1, -1]), "<- nút 3 bị đếm 2 lần nên count=5 ≠ 4, may mắn ra False")

ví dụ 3 (chu trình): > 50 bước (sẽ chạy mãi)
ví dụ 2 (nút 2 cha): False <- nút 3 bị đếm 2 lần nên count=5 ≠ 4, may mắn ra False


**Vì sao sai:** có chu trình thì hàng đợi không bao giờ rỗng. **Sửa:** kiểm tra bậc vào ≤ 1 và gốc duy nhất **trước** BFS (khi đó BFS chắc chắn dừng), hoặc dùng `seen`.

### Lỗi 4 — Union-Find chỉ kiểm tra chu trình, quên "hai cha"

In [23]:
def wrong_dsu_no_parent_check(n, L, R):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    comp = n
    for p in range(n):
        for c in (L[p], R[p]):
            if c == -1:
                continue
            rp, rc = find(p), find(c)
            if rp == rc:
                return False
            parent[rc] = rp
            comp -= 1
    return comp == 1

case = (3, [1, -1, 1], [-1, -1, -1])      # 0->1 và 2->1: nút 1 có hai cha
print("sai:", wrong_dsu_no_parent_check(*case), "| đúng:", validate_bfs(*case))

sai: True | đúng: False


**Vì sao sai:** đồ thị **vô hướng** `0 — 1 — 2` là một cây, nhưng theo **hướng** thì nút 1 có hai cha và có hai "gốc" 0, 2. **Sửa:** thêm `has_parent` (hoặc kiểm tra bậc vào).

### Lỗi 5 — Đếm cạnh `n − 1` thay cho kiểm tra liên thông

In [24]:
def wrong_edge_count(n, L, R):
    indeg = indegrees(n, L, R)
    return max(indeg) <= 1 and len(edges_of(n, L, R)) == n - 1

n, L, R, want = tricky
print("sai:", wrong_edge_count(n, L, R), "| đúng:", want)

sai: True | đúng: False


**Vì sao sai:** `n − 1` cạnh + bậc vào ≤ 1 vẫn có thể là "một cây + một chu trình" (ví dụ `tricky` có 3 cạnh với 4 nút). **Sửa:** kiểm tra liên thông bằng BFS hoặc Union-Find.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Các cạnh có tạo thành một cây không?"** ⇒ checklist ba điều: (1) mỗi nút ≤ 1 cha, (2) đúng một gốc, (3) liên thông từ gốc.
- Đồ thị **vô hướng**: cây ⇔ `n − 1` cạnh **và** liên thông (hoặc Union-Find không chu trình + 1 nhóm). Đồ thị **có hướng**: thêm điều kiện bậc vào.
- Không bao giờ giả định gốc là 0 — **tìm** gốc bằng bậc vào.
- BFS trên đồ thị có thể có chu trình ⇒ luôn có `seen` hoặc một điều kiện đảm bảo dừng.
- Khi viết test cho bài cây, có hàm **sinh cây hợp lệ ngẫu nhiên** + **đột biến** là cách bắt lỗi rất hiệu quả.

## 15. Biến thể của bài

1. **LeetCode 261 — Graph Valid Tree:** đồ thị vô hướng, kiểm tra cây ⇒ Union-Find hoặc `n − 1` cạnh + liên thông.
2. **LeetCode 684 / 685 — Redundant Connection (I / II):** tìm cạnh thừa làm hỏng cây (II là bản có hướng — xử lý hai cha + chu trình, rất gần bài này).
3. **Trả về gốc** (hoặc `−1` nếu không hợp lệ) — Bài 4.
4. **Cây `k`-phân** (mỗi nút có list con): thuật toán y hệt, chỉ đổi cách duyệt con.
5. **Trả về level-order** của cây nếu hợp lệ — Bài 5.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi gốc | DFS từ mỗi nút, phát hiện thăm lại | `O(n²)` | `O(n)` | baseline, trọng tài |
| 2. Bậc vào + gốc + BFS | ba kiểm tra rẻ | `O(n)` | `O(n)` | **nộp bài**, dễ giải thích |
| 3. Union-Find | hai cha / chu trình / số nhóm | `O(n α)` | `O(n)` | xử lý cạnh dạng dòng, bài 685 |
| 4. Bậc vào + DFS kích thước | `size(root) == n` | `O(n)` | `O(n)` | khi cần thêm thông tin cây con |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `validate(3, [1,-1,-1], [2,-1,-1])`, `validate(3, [-1,-1,-1], [-1,-1,-1])`, `validate(3, [1,0,-1], [-1,-1,-1])`.

**Bài 2 (tính tay):** Với `n = 5, leftChild = [4,-1,-1,-1,1], rightChild = [3,-1,-1,-1,2]` — tính `indeg`, gốc, thứ tự BFS và kết luận.

**Bài 3 (điền TODO):** Hoàn thiện `find_root(n, L, R)` trả về gốc duy nhất (nút bậc vào 0) hoặc `-1` nếu số gốc khác 1 hoặc có nút hai cha.

**Bài 4 (tự viết lại):** Viết lại Cách 3 (Union-Find) **có union theo kích thước** (union by size), không nhìn code mẫu.

**Bài 5 (mini challenge):** `level_order_if_valid(n, L, R)`: nếu hợp lệ trả về level-order (dạng LeetCode, có `None`, bỏ `None` cuối), ngược lại trả `[]`.

In [25]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def find_root(n, L, R):
    indeg = [0] * n
    # TODO: đếm bậc vào, nếu nút nào > 1 -> return -1
    # TODO: nếu đúng một nút bậc vào 0 -> trả nút đó, ngược lại -1
    pass

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def validate_dsu_by_size(n, L, R):
    # TODO: parent, size; union gắn cây nhỏ vào cây lớn
    pass

def level_order_if_valid(n, L, R):
    # TODO: dùng validate + find_root + build_tree + to_level_order
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Tính bậc vào. (2) Đếm số gốc. (3) Ca cuối: `0 ↔ 1` là chu trình, nút 2 là gốc riêng.

**Bài 2:** (1) Cạnh: `0→4, 0→3, 4→1, 4→2`. (2) `indeg = [0,1,1,1,1]`, gốc 0. (3) BFS: `0, 4, 3, 1, 2`.

**Bài 3:** (1) Một vòng đếm. (2) `indeg.count(0) == 1`. (3) `indeg.index(0)`.

**Bài 4:** (1) `size = [1]*n`. (2) Khi gộp, gắn gốc có size nhỏ vào gốc size lớn, cộng size. (3) Vẫn giữ `has_parent` và kiểm tra chu trình.

**Bài 5:** (1) `if not validate_bfs(...): return []`. (2) `root = find_root(...)`. (3) `to_level_order(build_tree(root, L, R))` — kiểm tra vòng bằng `build_from_level_order`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
r1 = [validate_bfs(3, [1, -1, -1], [2, -1, -1]), validate_bfs(3, [-1, -1, -1], [-1, -1, -1]), validate_bfs(3, [1, 0, -1], [-1, -1, -1])]
print("Bài 1:", r1)
assert r1 == [True, False, False]

# Bài 2
print("Bài 2:")
assert trace_bfs(5, [4, -1, -1, -1, 1], [3, -1, -1, -1, 2]) is True

Bài 1: [True, False, False]
Bài 2:
indeg = [0, 1, 1, 1, 1]
gốc = [0]
  thăm 0, con [4, 3], hàng đợi [4, 3], đã thăm 1
  thăm 4, con [1, 2], hàng đợi [3, 1, 2], đã thăm 2
  thăm 3, con [], hàng đợi [1, 2], đã thăm 3
  thăm 1, con [], hàng đợi [2], đã thăm 4
  thăm 2, con [], hàng đợi [], đã thăm 5
count=5, n=5 -> True


In [28]:
# Bài 3
def find_root_sol(n, L, R):
    indeg = [0] * n
    for c in L + R:
        if c != -1:
            indeg[c] += 1
            if indeg[c] > 1:
                return -1
    return indeg.index(0) if indeg.count(0) == 1 else -1

assert find_root_sol(4, [3, -1, 1, -1], [-1, -1, 0, -1]) == 2
assert find_root_sol(2, [1, 0], [-1, -1]) == -1
assert find_root_sol(4, [1, -1, 3, -1], [2, 3, -1, -1]) == -1
print("Bài 3: đúng")

# Bài 4
def validate_dsu_by_size_sol(n, L, R):
    parent, size = list(range(n)), [1] * n
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    has_parent, comp = [False] * n, n
    for p in range(n):
        for c in (L[p], R[p]):
            if c == -1:
                continue
            if has_parent[c]:
                return False
            has_parent[c] = True
            a, b = find(p), find(c)
            if a == b:
                return False
            if size[a] < size[b]:
                a, b = b, a
            parent[b] = a
            size[a] += size[b]
            comp -= 1
    return comp == 1

rng = random.Random(4)
for _ in range(800):
    n, L, R = random_valid_tree(rng, rng.randint(1, 9))
    if rng.random() < 0.7:
        arr = rng.choice([L, R]); arr[rng.randrange(n)] = rng.randint(-1, n - 1)
    assert validate_dsu_by_size_sol(n, L, R) == validate_brute(n, L, R)
print("Bài 4: khớp brute force trên 800 đầu vào")

Bài 3: đúng
Bài 4: khớp brute force trên 800 đầu vào


In [29]:
# Bài 5
def level_order_if_valid_sol(n, L, R):
    if not validate_bfs(n, L, R):
        return []
    return to_level_order(build_tree(find_root_sol(n, L, R), L, R))

print("Bài 5:", level_order_if_valid_sol(4, [1, -1, 3, -1], [2, -1, -1, -1]), level_order_if_valid_sol(*tricky[:3]))
assert level_order_if_valid_sol(4, [1, -1, 3, -1], [2, -1, -1, -1]) == [0, 1, 2, None, None, 3]
assert level_order_if_valid_sol(*tricky[:3]) == []
rng = random.Random(5)
for _ in range(200):
    n, L, R = random_valid_tree(rng, rng.randint(1, 10))
    lo = level_order_if_valid_sol(n, L, R)
    n2, L2, R2 = build_from_level_order(lo)
    assert (n2, L2, R2) == (n, L, R)          # dựng lại đúng cấu trúc ban đầu
print("Bài 5: level-order dựng lại đúng cây trên 200 cây ngẫu nhiên")

Bài 5: [0, 1, 2, None, None, 3] []
Bài 5: level-order dựng lại đúng cây trên 200 cây ngẫu nhiên


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
indeg = [0] * n
for c in leftChild + rightChild:
    if c != -1:
        indeg[c] += 1
        if indeg[c] > 1: return False          # hai cha
roots = [i for i in range(n) if indeg[i] == 0]
if len(roots) != 1: return False               # 0 hoặc nhiều gốc
q, cnt = deque(roots), 0
while q:
    u = q.popleft(); cnt += 1
    for c in (leftChild[u], rightChild[u]):
        if c != -1: q.append(c)
return cnt == n                                # liên thông
```

**Checklist trước khi submit:**
- [ ] Kiểm tra nút có **hai cha** (kể cả `leftChild[i] == rightChild[i]`).
- [ ] Tìm gốc bằng bậc vào, **không** giả định gốc là 0.
- [ ] Kiểm tra **liên thông** (thử ca chu trình tách rời `n=4, L=[1,0,3,-1]`).
- [ ] BFS chỉ chạy sau khi đã đảm bảo dừng (bậc vào ≤ 1, gốc duy nhất) hoặc có `seen`.
- [ ] `n = 1` ⇒ True; tự trỏ `L=[0]` ⇒ False.
- [ ] Đúng chữ ký: `validateBinaryTreeNodes(self, n, leftChild, rightChild) -> bool`.